# 04 — Rhythm Feature Extraction

Extract **rhythm** features per song (`song_id` aligned with the Stage 1 manifest).

On Kaggle, raw MP3s are optional. Default path: **approximate from log-mel** already on disk (Phase 2 practical path). Attach an audio dataset under `/kaggle/input/mtg-jamendo-audio` to use real librosa audio features.


## Kaggle setup (every notebook)

### A. Settings
1. Right sidebar → **Internet → On** (required for downloads).
2. **GPU**: Off for `00`/`01`/`04`–`06`. **GPU (T4)** on for `02`/`03`/`07`.

### B. How data moves (do not skip)
Kaggle **does not** keep `/kaggle/working` when you open a *new* notebook.

**After notebook 00 finishes:**
1. **Save Version** (top-right) → **Save & Run All** (or Quick Save if already finished).
2. Open **Advanced** → tick **Always save output**.
3. Wait until the version is **Success**.
4. Note the kernel slug (yours is **`thevifernando/dnn-download-data-1`**).

**In the next notebook (01, then 02, …):**
1. **Add Input** (right sidebar) → **Your notebooks** / **Notebook Output**.
2. Select **`dnn-download-data-1`** (latest successful version).
3. Files appear at `/kaggle/input/dnn-download-data-1/` (**read-only**).
4. This bootstrap **reads mels from that input** (does **not** copy 10 shards — they would overflow disk).
5. It **writes** new files (manifest, features, checkpoints) to `/kaggle/working/MTG_Instrument`.
6. **Save Version + save output** again so the *next* notebook can **Add Input** *this* notebook too (chain: 00 → 01 → 02 …).

### C. CLI (laptop only — not needed on Kaggle)
```bash
kaggle kernels output thevifernando/dnn-download-data-1 -p ./from_00
```
On Kaggle you **Add Input** instead of this command.

### D. GitHub
Commit **notebooks only** to `thevindu-branch`. Do **not** git-push the `.npy` shards (too large). Data stays on Kaggle output.


## Step 0 — Packages (`librosa` is slow on CPU; that is OK for this notebook)


In [ ]:
!pip install -q librosa soundfile tqdm


## Step 1 — Bootstrap paths


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, time, urllib.request
import numpy as np
import pandas as pd

KERNEL_SLUG = "dnn-download-data-1"  # notebook 00 Kaggle slug — change if yours differs
WORKING_ROOT = Path("/kaggle/working/MTG_Instrument")
INPUT_BASE = Path("/kaggle/input")
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host: str = "github.com", port: int = 443, timeout: float = 5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    """MTG ids are 7-digit zero-padded (track_0000948 → 0000948)."""
    m = re.search(r"(\d+)", str(raw))
    if not m:
        return None
    return f"{int(m.group(1)):07d}"


def _find_file(name: str, bases: list[Path]) -> Path | None:
    for base in bases:
        if not base.exists():
            continue
        hits = list(base.rglob(name))
        if hits:
            return hits[0]
    return None


def discover_input_root() -> Path | None:
    """Find a previous notebook-00 output or MTG data folder under /kaggle/input."""
    if not INPUT_BASE.exists():
        return None
    for marker in [
        "song_manifest.csv",
        "autotagging_genre-train.tsv",
        "autotagging_genre.tsv",
        ".shard_00_done",
    ]:
        hit = _find_file(marker, [INPUT_BASE])
        if hit is None:
            continue
        if marker == "song_manifest.csv":
            return hit.parents[1]  # .../MTG_Instrument/dataset/song_manifest.csv
        if marker == "autotagging_genre-train.tsv":
            # .../annotations/splits/split-0/file  OR  .../data/splits/split-0/file
            p = hit
            for _ in range(6):
                if (p / "dataset").exists() or p.name in {"MTG_Instrument", "data"}:
                    return p if p.name != "data" else p
                p = p.parent
            return hit.parents[2]
        if marker == "autotagging_genre.tsv":
            parent = hit.parent
            if parent.name == "annotations":
                return parent.parent
            return parent  # MTG data/
        if marker == ".shard_00_done":
            return hit.parents[2]  # .../MTG_Instrument/dataset/logmel_songs/.shard
    for p in INPUT_BASE.rglob("MTG_Instrument"):
        if p.is_dir():
            return p
    return None


def find_mel_dir() -> Path:
    """Prefer attached kernel output (read-only). Never copy 10 shards into working."""
    bases = [
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "dataset" / "logmel_songs",
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "dataset" / "logmel_songs",
        WORKING_ROOT / "dataset" / "logmel_songs",
    ]
    kernel = Path(f"/kaggle/input/{KERNEL_SLUG}")
    extra = []
    if INPUT_BASE.exists():
        extra.append(INPUT_BASE)
    if kernel.exists():
        extra.append(kernel)
    for b in bases:
        if b.exists() and next(b.rglob("*.npy"), None) is not None:
            return b
    for b in extra:
        hit = next(b.rglob("*.npy"), None) if b.exists() else None
        if hit is None:
            continue
        p = hit.parent
        for _ in range(6):
            if p.name == "logmel_songs":
                return p
            p = p.parent
        return hit.parent
    return WORKING_ROOT / "dataset" / "logmel_songs"


def ensure_annotations(ann_dir: Path) -> Path:
    """Make sure split-0 TSVs exist; wget them if this is a fresh Kaggle session."""
    train = ann_dir / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if train.exists():
        return ann_dir

    # maybe files are flat, or under /kaggle/input with a different layout
    hit = _find_file("autotagging_genre-train.tsv", [ann_dir, INPUT_BASE, Path("/kaggle/working")])
    if hit is not None:
        dest = ann_dir / "splits" / "split-0" / hit.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if hit.resolve() != dest.resolve():
            shutil.copy2(hit, dest)
        # copy sibling split files from the same folder
        for name in [
            "autotagging_genre-validation.tsv",
            "autotagging_genre-test.tsv",
            "autotagging_instrument-train.tsv",
            "autotagging_instrument-validation.tsv",
            "autotagging_instrument-test.tsv",
        ]:
            sib = hit.parent / name
            if sib.exists():
                shutil.copy2(sib, dest.parent / name)
        genre_full = _find_file("autotagging_genre.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if genre_full:
            shutil.copy2(genre_full, ann_dir / "autotagging_genre.tsv")
        inst_full = _find_file("autotagging_instrument.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if inst_full:
            shutil.copy2(inst_full, ann_dir / "autotagging_instrument.tsv")
        print("Recovered split files from", hit.parent)
        return ann_dir

    if not check_internet():
        raise FileNotFoundError(
            "Split TSVs not found and Internet is OFF.\n"
            "Do ONE of:\n"
            "  A) Settings → Internet → On, re-run this cell (auto-download)\n"
            "  B) Add Data → attach notebook-00 output dataset (mtg-instrument-cache)\n"
            "  C) Stay in the SAME Kaggle session after running notebook 00"
        )

    print("Split TSVs missing — downloading official MTG annotations...")
    n = 0
    for rel in NEEDED_ANN:
        dest = ann_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"{RAW_ANN}/{rel}"
        print("  wget", url)
        urllib.request.urlretrieve(url, dest)
        n += 1
    print(f"Downloaded {n} annotation files into {ann_dir}")
    return ann_dir


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    candidates = [
        ANN_DIR / "splits" / "split-0" / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / "splits" / "split-0" / f"{split}.tsv",
        ANN_DIR / f"{split}.tsv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        found = _find_file(f"autotagging_{subset}-{split}.tsv", [ANN_DIR, INPUT_BASE, Path("/kaggle/working")])
        path = found
    if path is None:
        raise FileNotFoundError(
            f"No split file for {subset}/{split}.\n"
            "Re-run the bootstrap cell after enabling Internet, or attach notebook-00 output."
        )
    df = pd.read_csv(path, sep="\t")
    col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    ids = set()
    for v in df[col].astype(str):
        tid = normalize_track_id(v)
        if tid:
            ids.add(tid)
    print(f"{split:12s}  {len(ids):6d} ids   ← {path}")
    return ids


MEL_CACHE = Path("/kaggle/working/mel_cache")
MEL_CACHE.mkdir(parents=True, exist_ok=True)


def load_mel_npy(mel_abs, retries=5, pause=1.0):
    """Load mel with retries; cache under /kaggle/working for stable re-reads."""
    mel_abs = Path(mel_abs)
    sid = normalize_track_id(mel_abs.stem) or mel_abs.stem.replace("/", "_")
    cached = MEL_CACHE / f"{sid}.npy"
    if cached.exists():
        try:
            return np.load(cached)
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)

    last_err = None
    for attempt in range(retries):
        try:
            arr = np.load(mel_abs, mmap_mode=None)
            arr = np.asarray(arr, dtype=np.float32)
            np.save(cached, arr)
            return arr
        except (OSError, ValueError) as e:
            last_err = e
            if attempt + 1 < retries:
                time.sleep(pause * (attempt + 1))
    nbytes = mel_abs.stat().st_size if mel_abs.exists() else "missing"
    raise RuntimeError(
        f"Bad/truncated mel — re-run notebook 00 for this shard: {mel_abs} "
        f"({nbytes} bytes). {last_err}"
    ) from last_err


def scan_bad_mels(df, label="manifest"):
    from tqdm.auto import tqdm

    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"scan {label}"):
        try:
            load_mel_npy(row["mel_abs"])
        except Exception as e:
            bad.append({"song_id": str(row["song_id"]), "mel_abs": row["mel_abs"], "error": str(e)})
    if bad:
        out = RESULTS_DIR / f"bad_mels_{label}.json"
        out.write_text(json.dumps(bad, indent=2))
        print(f"WARNING: {len(bad)} bad mels → {out}")
    else:
        print(f"scan {label}: all {len(df)} mels OK (cache: {MEL_CACHE})")
    return bad


ONLINE = check_internet()
print("Internet reachable:", ONLINE)
print("KERNEL_SLUG =", KERNEL_SLUG)
print("/kaggle/input folders:", list(INPUT_BASE.iterdir()) if INPUT_BASE.exists() else "n/a")

ROOT = WORKING_ROOT
ROOT.mkdir(parents=True, exist_ok=True)
MEL_DIR = find_mel_dir()
ANN_DIR = ROOT / "annotations"
# if annotations only exist on the attached kernel, point there (read-only is OK)
for cand in [
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "annotations",
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "annotations",
]:
    if (cand / "splits" / "split-0" / "autotagging_genre-train.tsv").exists():
        ANN_DIR = cand
        break
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
att_manifest = _find_file("song_manifest.csv", [INPUT_BASE, Path("/kaggle/working")])
if not MANIFEST.exists() and att_manifest is not None:
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copy2(att_manifest, MANIFEST)
        print("Copied song_manifest.csv from", att_manifest)
    except OSError:
        MANIFEST = att_manifest

for p in [ROOT / "dataset", ROOT / "annotations", FEAT_DIR, CKPT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Small TSVs: copy/wget into working. Large mels stay on /kaggle/input.
ANN_DIR = ensure_annotations(ROOT / "annotations")

print("ROOT     =", ROOT)
print("MEL_DIR  =", MEL_DIR, "npy=", len(list(MEL_DIR.rglob('*.npy'))))
print("ANN_DIR  =", ANN_DIR)
print("split-0 train exists:", (ANN_DIR / "splits/split-0/autotagging_genre-train.tsv").exists())
print("MANIFEST =", MANIFEST, "exists=", MANIFEST.exists())


## Step 2 — Extract rhythm and write `features/rhythm/rhythm_song.csv`


In [ ]:
import librosa
from tqdm.auto import tqdm

if not MANIFEST.exists():
    raise FileNotFoundError("Run notebook 01 first.")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
AUDIO_ROOT = Path("/kaggle/input/mtg-jamendo-audio")
WINDOW_SEC = 15.0
SR = 22050


def extract_from_audio(path: Path) -> dict:
    y, sr = librosa.load(path, sr=SR, mono=True, duration=60)
    hop = int(WINDOW_SEC * sr)
    vals = []
    for start in range(0, max(len(y) - hop, 0) + 1, hop):
        yw = y[start:start + hop]
        if len(yw) < hop // 2:
            break
        onset_env = librosa.onset.onset_strength(y=yw, sr=sr)
        tempo, beats = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr)
        bt = librosa.frames_to_time(beats, sr=sr) if len(beats) else np.array([])
        intervals = np.diff(bt) if len(bt) > 1 else np.array([np.nan])
        vals.append(dict(
            tempo=float(np.atleast_1d(tempo)[0]),
            beat_strength_mean=float(np.mean(onset_env)),
            onset_density=float(len(librosa.onset.onset_detect(y=yw, sr=sr)) / WINDOW_SEC),
            beat_interval_mean=float(np.nanmean(intervals)),
            beat_interval_std=float(np.nanstd(intervals)),
        ))
    return pd.DataFrame(vals).mean(numeric_only=True).to_dict() if vals else {}

def mel_proxy_features(S: np.ndarray) -> dict:
    env = S.mean(axis=0)
    env = (env - env.mean()) / (env.std() + 1e-6)
    return dict(
        tempo=float(60.0),
        beat_strength_mean=float(np.mean(np.abs(env))),
        onset_density=float(np.mean(env > 1.0)),
        beat_interval_mean=float("nan"),
        beat_interval_std=float("nan"),
    )


def features_from_mel(mel_path: Path) -> dict:
    S = load_mel_npy(mel_path)
    if S.ndim == 3:
        S = S.mean(0)
    return mel_proxy_features(S)

rows = []
for _, rec in tqdm(manifest.iterrows(), total=len(manifest)):
    sid = str(rec["song_id"])
    audio_candidate = None
    if "audio_path" in manifest.columns and pd.notna(rec.get("audio_path", None)):
        audio_candidate = Path(rec["audio_path"])
    elif AUDIO_ROOT.exists():
        hits = list(AUDIO_ROOT.rglob(f"*{sid}*.mp3"))
        audio_candidate = hits[0] if hits else None
    try:
        if audio_candidate and Path(audio_candidate).exists():
            feat = extract_from_audio(Path(audio_candidate))
            src = "audio"
        else:
            feat = features_from_mel(Path(rec["mel_abs"]))
            src = "mel_proxy"
    except Exception as e:
        print("fail", sid, e)
        continue
    feat.update({"song_id": sid, "source": src, "split": rec["split"]})
    rows.append(feat)

df = pd.DataFrame(rows)
out = FEAT_DIR / "rhythm"
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "rhythm_song.csv", index=False)
print(df.head())
print("wrote", out, "n=", len(df))
